# Finite-Geometry AA-DPD PET Slab for PETase Interface Models

This notebook builds a role-aware all-atom PET dense periodic cell with MuPT, using an 8 x 8 x 2 nm box at the target PET density as the initialization geometry. The PBC vectors are only an initialization aid: the exported whole-chain CIF provides sensible PET slab coordinates for later protein/water setup workflows.

The target application is PETase at a water-bottle PET interface. The slab density target is 1.38 g/cm^3, and the lateral dimensions are 8 x 8 nm so a roughly 6 x 6 x 6 nm enzyme can be visually checked against a surface that is wider than the enzyme footprint.


## 1. Environment and Science Knobs

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import os
import sys
import time

import networkx as nx
import numpy as np
from anytree import PreOrderIter
from rdkit import Chem
from rdkit.Geometry import Point3D

def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")

EXAMPLES_ROOT = find_examples_root()
MUPT_SOURCE_CANDIDATES = [
    Path(os.environ["MUPT_SOURCE"]).expanduser() if os.environ.get("MUPT_SOURCE") else None,
    EXAMPLES_ROOT.parent / "mupt",
    EXAMPLES_ROOT / "mupt",
]
for candidate in MUPT_SOURCE_CANDIDATES:
    if candidate is not None and (candidate / "mupt" / "builders" / "all_atom_dpd.py").exists():
        sys.path.insert(0, str(candidate))
        break

import mupt

from mupt.builders.all_atom_dpd import AllAtomDPDBuilder, AllAtomDPDSettings
from mupt.interfaces.rdkit import primitive_to_rdkit_mols
from mupt.interfaces.smiles import primitive_from_smiles
from mupt.mupr.primitives import Primitive
from mupt.mupr.topology import TopologicalStructure
from mupt.roles import PrimitiveRole
from mupt.temporary.sdf import prepare_mupt_sdf_atom_props

OUTPUT_ROOT = EXAMPLES_ROOT / "examples_system" / "finite_geometry_pet_slab_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 51
TARGET_DENSITY_G_CM3 = 1.38
SLAB_X_NM = 8.0
SLAB_Y_NM = 8.0
SLAB_Z_NM = 2.0
DPD_BOX_Z_NM = SLAB_Z_NM
CHAIN_LENGTH_MIN = 10
CHAIN_LENGTH_MAX = 14
RUN_AA_DPD = True
RUN_OPENMM_MINIMIZATION = False  # OpenMM is useful but slow; enable after inspecting the exported SDF/CIF.

DPD_STEPS_MAX = 50_000
DPD_STEPS_PER_INTERVAL = 1_000
DPD_DT = 0.001
DPD_R_CUT_A = 3.0
DPD_A = 5_000.0
DPD_GAMMA = 800.0
DPD_KT = 1.0
DPD_BOND_K_SCALE = 1.0
DPD_ANGLE_K_SCALE = 1.0
DPD_DIHEDRAL_K_SCALE = 1.0  # Also applied to impropers by AllAtomDPDBuilder.
DPD_INITIAL_RESIDUE_SPACING_A = 1.45
WRITE_DPD_GSD = False

FORCE_FIELD = "openff-2.2.1.offxml"
PARTIAL_CHARGE_METHOD = "openff-gnn-am1bcc-1.0.0.pt"
OPENMM_PLATFORM_NAME = "CUDA"
OPENMM_PLATFORM_PROPERTIES = {"Precision": "mixed", "DeviceIndex": "0"}
OPENMM_MINIMIZATION_TOLERANCE_KJ_MOL_NM = 1.0e-3
OPENMM_MD_TEMPERATURE_K = 300.0
OPENMM_MD_FRICTION_PER_PS = 1.0
OPENMM_MD_TIMESTEP_FS = 2.0
OPENMM_MD_DURATION_NS = 0.5
OPENMM_MD_REPORT_INTERVAL_STEPS = 10_000
OPENMM_NPT_DURATION_NS = 0.5
OPENMM_NPT_PRESSURE_ATM = 1.0
OPENMM_BAROSTAT_FREQUENCY_STEPS = 25
TRAJECTORY_FRAMES = 150

DA_PER_NM3_TO_G_CM3 = 1.0 / 602.214076
slab_volume_nm3 = SLAB_X_NM * SLAB_Y_NM * SLAB_Z_NM
target_mass_da = TARGET_DENSITY_G_CM3 * slab_volume_nm3 / DA_PER_NM3_TO_G_CM3
PET_REPEAT_MASS_DA = 192.168  # C10H8O4 repeat mass, used only for chain-count planning
DPD_BOX_LENGTHS_A = (10.0 * SLAB_X_NM, 10.0 * SLAB_Y_NM, 10.0 * DPD_BOX_Z_NM)
chain_plan = AllAtomDPDBuilder.plan_uniform_chain_lengths_for_box(
    density_g_cm3=TARGET_DENSITY_G_CM3,
    box_lengths_a=DPD_BOX_LENGTHS_A,
    repeat_unit_mass_amu=PET_REPEAT_MASS_DA,
    chain_length_min=CHAIN_LENGTH_MIN,
    chain_length_max=CHAIN_LENGTH_MAX,
    random_seed=RANDOM_SEED,
)
N_CHAINS = len(chain_plan.chain_lengths)
OPENMM_MD_STEPS = int(round(OPENMM_MD_DURATION_NS * 1_000_000.0 / OPENMM_MD_TIMESTEP_FS))
OPENMM_NPT_STEPS = int(round(OPENMM_NPT_DURATION_NS * 1_000_000.0 / OPENMM_MD_TIMESTEP_FS))

print(f"Repository root: {EXAMPLES_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"MuPT import path: {mupt.__file__}")
print(f"Target slab: {SLAB_X_NM} x {SLAB_Y_NM} x {SLAB_Z_NM} nm^3")
print(f"Target PET mass: {chain_plan.target_mass_amu:.1f} Da")
print(f"Planned PET mass: {chain_plan.planned_mass_amu:.1f} Da")
print(f"Using {N_CHAINS} chains with repeat-unit lengths in [{CHAIN_LENGTH_MIN}, {CHAIN_LENGTH_MAX}]")
print(f"First 10 chain lengths: {chain_plan.chain_lengths[:10]}")

## 2. Build Role-Aware PET Chains in MuPT

PET is represented as hydroxyl/carboxyl terminated oligomers with connector-marked head, middle, and tail residues. The `*` atoms mark polymerization linkers; atom-map labels give MuPT a consistent traversal direction for topology registration.

In [ ]:
PET_RESIDUE_SMILES = {
    "head": "[H]O[CH2][CH2]OC(=O)c1ccc([C:2](=O)-*)cc1",
    "mid": "*-[O:1][CH2][CH2]OC(=O)c1ccc([C:2](=O)-*)cc1",
    "tail": "*-[O:1][CH2][CH2]OC(=O)c1ccc([C:2](=O)O[H])cc1",
}
RESNAME_MAP = {"head": "PTH", "mid": "PET", "tail": "PTT"}

@dataclass(frozen=True)
class BuiltPETSlab:
    primitive: Primitive
    chain_sequences: list[list[str]]
    total_mass_da: float

def build_pet_lexicon() -> dict[str, Primitive]:
    lexicon = {}
    for name, smiles in PET_RESIDUE_SMILES.items():
        residue = primitive_from_smiles(smiles, ensure_explicit_Hs=True, embed_positions=True, label=name)
        residue.role = PrimitiveRole.RESIDUE
        residue.metadata["residue_name"] = RESNAME_MAP[name]
        for atom in residue.children:
            atom.role = PrimitiveRole.PARTICLE
        lexicon[name] = residue
    return lexicon

def primitive_mass_da(root: Primitive) -> float:
    mass = 0.0
    for atom in root.leaves:
        if atom.element is None:
            raise ValueError(f"Atomic primitive {atom.label!r} has no element")
        mass += float(atom.element.mass)
    return mass

def pet_chain_sequence(repeat_units: int) -> list[str]:
    if repeat_units < 2:
        raise ValueError("PET chain lengths must be at least 2 for head/tail residues")
    return ["head", *(["mid"] * (repeat_units - 2)), "tail"]


def build_pet_slab(chain_lengths: list[int]) -> BuiltPETSlab:
    lexicon = build_pet_lexicon()
    universe = Primitive(label="pet_water_bottle_interface_slab", role=PrimitiveRole.UNIVERSE)
    universe.metadata.update({
        "system_name": "PET_slab_8x8x2nm",
        "target_density_g_cm3": str(TARGET_DENSITY_G_CM3),
        "slab_dimensions_nm": json.dumps([SLAB_X_NM, SLAB_Y_NM, SLAB_Z_NM]),
        "dpd_box_dimensions_nm": json.dumps([SLAB_X_NM, SLAB_Y_NM, DPD_BOX_Z_NM]),
        "placement_method": "AllAtomDPDBuilder(box_lengths_a)",
        "chain_length_distribution": "uniform_min_max",
        "chain_length_min": str(CHAIN_LENGTH_MIN),
        "chain_length_max": str(CHAIN_LENGTH_MAX),
    })
    chain_sequences = []
    for chain_idx, repeat_units in enumerate(chain_lengths):
        segment = Primitive(label=f"pet_chain_{chain_idx:04d}", role=PrimitiveRole.SEGMENT)
        sequence = pet_chain_sequence(repeat_units)
        chain_sequences.append(sequence)
        handles = []
        for repeat_idx, residue_name in enumerate(sequence):
            residue = lexicon[residue_name].copy()
            residue.role = PrimitiveRole.RESIDUE
            residue.label = f"{residue_name}_{repeat_idx:03d}"
            residue.metadata.update({
                "residue_name": RESNAME_MAP[residue_name],
                "chain_index": str(chain_idx),
                "repeat_index": str(repeat_idx),
            })
            for atom in residue.children:
                atom.role = PrimitiveRole.PARTICLE
            handles.append(segment.attach_child(residue))
        segment.set_topology(nx.path_graph(handles, create_using=TopologicalStructure), max_registration_iter=100)
        universe.attach_child(segment)
    total_mass = primitive_mass_da(universe)
    universe.metadata["total_mass_da"] = str(total_mass)
    universe.metadata["actual_initial_density_g_cm3"] = str(total_mass * DA_PER_NM3_TO_G_CM3 / slab_volume_nm3)
    return BuiltPETSlab(primitive=universe, chain_sequences=chain_sequences, total_mass_da=total_mass)

built = build_pet_slab(chain_plan.chain_lengths)
universe = built.primitive
print(universe.hierarchy_summary(to_depth=2))
print(f"Mass: {built.total_mass_da:.1f} Da")
print(f"Slab density from chosen chains: {built.total_mass_da * DA_PER_NM3_TO_G_CM3 / slab_volume_nm3:.3f} g/cm^3")
print(f"Repeat-unit count range: {min(map(len, built.chain_sequences))} to {max(map(len, built.chain_sequences))}")

## 3. Finite-Geometry AA-DPD Slab Initialization

This is the main teaching step: MuPT already holds a role-aware hierarchy, so the source `AllAtomDPDBuilder` only needs settings. Supplying `box_lengths_a` selects the finite-geometry path instead of the default density-derived cubic box. The density was used above to choose how many chains to build; the builder now relaxes those chains inside the explicit 8 x 8 x 2 nm slab box.


In [ ]:
def residue_resname_map(root: Primitive) -> dict[str, str]:
    """Return the residue-name map used by MuPT -> RDKit/OpenFF export."""
    return {
        node.label: node.metadata.get("residue_name", "PET")
        for node in PreOrderIter(root)
        if getattr(node, "role", None) == PrimitiveRole.RESIDUE
    }




def rdkit_mols_from_universe(root: Primitive) -> list[Chem.Mol]:
    """Export each MuPT segment as an RDKit molecule with current atom coordinates."""
    return list(
        primitive_to_rdkit_mols(
            root,
            resname_map=residue_resname_map(root),
            default_atom_position=np.zeros(3),
        )
    )
def run_finite_geometry_aa_dpd(root: Primitive) -> list[float]:
    """Relax the PET hierarchy in the explicit slab box with AllAtomDPDBuilder."""
    dpd_output_name = None
    if WRITE_DPD_GSD:
        dpd_dir = OUTPUT_ROOT / "aa_dpd"
        dpd_dir.mkdir(parents=True, exist_ok=True)
        dpd_output_name = str(dpd_dir / "pet_slab_aa_dpd")

    settings = AllAtomDPDSettings(
        density_g_cm3=TARGET_DENSITY_G_CM3,
        box_lengths_a=DPD_BOX_LENGTHS_A,
        r_cut_a=DPD_R_CUT_A,
        kT=DPD_KT,
        A_base=DPD_A,
        gamma_base=DPD_GAMMA,
        dt=DPD_DT,
        particle_spacing_a=0.75,
        initial_residue_spacing_a=DPD_INITIAL_RESIDUE_SPACING_A,
        n_steps_max=DPD_STEPS_MAX,
        n_steps_per_interval=DPD_STEPS_PER_INTERVAL,
        report_interval=DPD_STEPS_PER_INTERVAL,
        force_field=FORCE_FIELD,
        bond_scale=DPD_BOND_K_SCALE,
        angle_scale=DPD_ANGLE_K_SCALE,
        dihedral_scale=DPD_DIHEDRAL_K_SCALE,
        random_seed=RANDOM_SEED,
        write_gsd=WRITE_DPD_GSD,
        output_name=dpd_output_name,
        resname_map=residue_resname_map(root),
    )
    result = AllAtomDPDBuilder(settings=settings).build(root)
    summary = root.metadata["all_atom_dpd_summary"]
    print("AA-DPD summary")
    print(f"  atoms: {summary['n_atoms']}")
    print(f"  bonds / angles / dihedrals / impropers: {summary['n_bonds']} / {summary['n_angles']} / {summary['n_dihedrals']} / {summary['n_impropers']}")
    print(f"  box_lengths_a: {summary['box_lengths_a']}")
    print(f"  converged: {summary['converged']} after {summary['steps']} DPD steps")
    return root.metadata["unit_cell_parameters"]


if RUN_AA_DPD:
    box_parameters = run_finite_geometry_aa_dpd(universe)
else:
    box_parameters = [*DPD_BOX_LENGTHS_A, 90.0, 90.0, 90.0]
    universe.metadata["unit_cell_parameters"] = box_parameters
    print("AA-DPD skipped; exporting unrelaxed RDKit conformer coordinates.")

print(f"DPD/OpenMM box parameters in A: {box_parameters}")


## 4. Export SDF and Pre-Minimization CIF

In [ ]:
def make_rdkit_molecule_whole(mol: Chem.Mol, box_parameters_a: list[float] | tuple[float, ...] | None, centered_input: bool = True) -> None:
    """Unwrap each bonded molecule across an orthorhombic periodic box in-place."""
    if box_parameters_a is None or mol.GetNumAtoms() <= 1:
        return
    box_lengths = np.asarray(box_parameters_a[:3], dtype=float)
    if box_lengths.shape != (3,) or np.any(box_lengths <= 0) or not np.all(np.isfinite(box_lengths)):
        return
    conf = mol.GetConformer()
    wrapped_positions = np.asarray(conf.GetPositions(), dtype=float)
    unwrapped_positions = wrapped_positions.copy()
    adjacency = [[] for _ in range(mol.GetNumAtoms())]
    for bond in mol.GetBonds():
        begin_idx = bond.GetBeginAtomIdx()
        end_idx = bond.GetEndAtomIdx()
        adjacency[begin_idx].append(end_idx)
        adjacency[end_idx].append(begin_idx)
    visited = np.zeros(mol.GetNumAtoms(), dtype=bool)
    for root_idx in range(mol.GetNumAtoms()):
        if visited[root_idx]:
            continue
        visited[root_idx] = True
        stack = [root_idx]
        while stack:
            atom_idx = stack.pop()
            for neighbor_idx in adjacency[atom_idx]:
                if visited[neighbor_idx]:
                    continue
                delta = wrapped_positions[neighbor_idx] - wrapped_positions[atom_idx]
                delta -= np.round(delta / box_lengths) * box_lengths
                unwrapped_positions[neighbor_idx] = unwrapped_positions[atom_idx] + delta
                visited[neighbor_idx] = True
                stack.append(neighbor_idx)
    # Keep each molecule whole, then choose the periodic image whose lower
    # coordinate corner lies in the displayed 0..L cell for PyMOL.
    unwrapped_positions = unwrapped_positions + (0.5 * box_lengths if centered_input else 0.0)
    unwrapped_positions -= np.floor(unwrapped_positions.min(axis=0) / box_lengths) * box_lengths
    for atom_idx, position in enumerate(unwrapped_positions):
        conf.SetAtomPosition(atom_idx, Point3D(float(position[0]), float(position[1]), float(position[2])))

def atom_chain_id(atom: Chem.Atom) -> str:
    if atom.HasProp("chain_id"):
        return str(atom.GetProp("chain_id"))
    info = atom.GetPDBResidueInfo()
    return info.GetChainId().strip() if info is not None else "A"

def atom_residue_id(atom: Chem.Atom) -> str:
    if atom.HasProp("residue_id"):
        return str(atom.GetIntProp("residue_id"))
    info = atom.GetPDBResidueInfo()
    return str(info.GetResidueNumber()) if info is not None else "1"

def write_rdkit_mols_to_pdbx(rdkit_mols: list[Chem.Mol], output_path: Path, box_vectors_nm: np.ndarray | None = None) -> None:
    import openmm
    from openmm import unit as omm_unit
    from openmm.app import PDBxFile, Topology, element
    topology = Topology()
    positions = []
    chain_cache = {}
    residue_cache = {}
    for mol_idx, mol in enumerate(rdkit_mols):
        atom_lookup = {}
        conf = mol.GetConformer()
        for atom_idx, atom in enumerate(mol.GetAtoms()):
            chain_id = atom_chain_id(atom)
            chain = chain_cache.get(chain_id)
            if chain is None:
                chain = topology.addChain(chain_id)
                chain_cache[chain_id] = chain
            residue_name = atom.GetProp("residue_name") if atom.HasProp("residue_name") else "PET"
            residue_id = atom_residue_id(atom)
            residue_key = (chain_id, residue_id, residue_name)
            residue = residue_cache.get(residue_key)
            if residue is None:
                residue = topology.addResidue(residue_name, chain, id=residue_id)
                residue_cache[residue_key] = residue
            symbol = atom.GetSymbol()
            atom_lookup[atom_idx] = topology.addAtom(f"{symbol}{atom_idx + 1}", element.get_by_symbol(symbol), residue)
            p = conf.GetAtomPosition(atom_idx)
            positions.append(openmm.Vec3(p.x * 0.1, p.y * 0.1, p.z * 0.1))
        for bond in mol.GetBonds():
            topology.addBond(atom_lookup[bond.GetBeginAtomIdx()], atom_lookup[bond.GetEndAtomIdx()])
    if box_vectors_nm is not None:
        topology.setPeriodicBoxVectors(box_vectors_nm * omm_unit.nanometer)
    with output_path.open("w") as handle:
        PDBxFile.writeFile(topology, positions * omm_unit.nanometer, handle)

sdf_dir = OUTPUT_ROOT / "sdf"
sdf_dir.mkdir(parents=True, exist_ok=True)
rdkit_mols = rdkit_mols_from_universe(universe)
for mol in rdkit_mols:
    make_rdkit_molecule_whole(mol, box_parameters)
sdf_path = sdf_dir / "pet_slab_8x8x2nm_aa_dpd.sdf"
writer = Chem.SDWriter(str(sdf_path))
for mol in rdkit_mols:
    prepare_mupt_sdf_atom_props(mol)
    writer.write(mol)
writer.close()
box_vectors_nm = np.diag([SLAB_X_NM, SLAB_Y_NM, DPD_BOX_Z_NM])
pre_min_cif_path = OUTPUT_ROOT / "pet_slab_8x8x2nm_aa_dpd_pre_openmm.cif"
write_rdkit_mols_to_pdbx(rdkit_mols, pre_min_cif_path, box_vectors_nm=box_vectors_nm)
manifest = {
    "target_density_g_cm3": TARGET_DENSITY_G_CM3,
    "actual_chain_density_g_cm3": built.total_mass_da * DA_PER_NM3_TO_G_CM3 / slab_volume_nm3,
    "slab_dimensions_nm": [SLAB_X_NM, SLAB_Y_NM, SLAB_Z_NM],
    "dpd_openmm_box_nm": [SLAB_X_NM, SLAB_Y_NM, DPD_BOX_Z_NM],
    "n_chains": N_CHAINS,
    "chain_length_min": CHAIN_LENGTH_MIN,
    "chain_length_max": CHAIN_LENGTH_MAX,
    "chain_lengths": chain_plan.chain_lengths,
    "target_mass_amu": chain_plan.target_mass_amu,
    "planned_mass_amu": chain_plan.planned_mass_amu,
    "all_atom_dpd_summary": universe.metadata.get("all_atom_dpd_summary"),
    "sdf": str(sdf_path.relative_to(EXAMPLES_ROOT)),
    "pre_min_cif": str(pre_min_cif_path.relative_to(EXAMPLES_ROOT)),
}
manifest_path = OUTPUT_ROOT / "pet_slab_8x8x2nm_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Wrote {len(rdkit_mols)} PET chain records to {sdf_path}")
print(f"Wrote pre-OpenMM CIF to {pre_min_cif_path}")
print(f"Wrote manifest to {manifest_path}")

## 5. OpenFF/OpenMM Minimization and Final CIF

In [ ]:
if RUN_OPENMM_MINIMIZATION:
    from openff.interchange import Interchange
    from openff.toolkit import ForceField, Molecule, Topology
    from openff.toolkit.utils import ToolkitRegistry
    from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper
    from openff.units import unit as off_unit
    import openmm
    from openmm import LangevinMiddleIntegrator, MonteCarloBarostat
    from openmm import unit as omm_unit
    from openmm.app import DCDFile, Simulation

    def transfer_metadata(rdkit_mol: Chem.Mol, off_mol: Molecule) -> None:
        for rd_atom, off_atom in zip(rdkit_mol.GetAtoms(), off_mol.atoms):
            props = rd_atom.GetPropsAsDict(includePrivate=True, includeComputed=False)
            off_atom.metadata.update({
                "residue_name": str(props.get("residue_name", "PET")),
                "residue_number": str(props.get("residue_id", props.get("mupt_residue_index", "1"))),
                "chain_id": str(props.get("chain_id", "A")),
                "atom_name": f"{rd_atom.GetSymbol()}{rd_atom.GetIdx() + 1}",
            })

    def build_openff_topology_from_instances(rdkit_mols: list[Chem.Mol]):
        instance_molecules = []
        positions = []
        for rdkit_mol in rdkit_mols:
            off_mol = Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True, hydrogens_are_explicit=True)
            transfer_metadata(rdkit_mol, off_mol)
            instance_molecules.append(off_mol)
            positions.append(off_mol.conformers[0].m_as(off_unit.angstrom))
        template = instance_molecules[0]
        if NAGLToolkitWrapper.is_available():
            template.assign_partial_charges(
                partial_charge_method=PARTIAL_CHARGE_METHOD,
                toolkit_registry=ToolkitRegistry([NAGLToolkitWrapper()]),
            )
        else:
            raise RuntimeError("OpenFF NAGL is required for fast PET charge assignment in this notebook")
        topology = Topology.from_molecules([template] * len(instance_molecules))
        return topology, np.vstack(positions) * off_unit.angstrom, [template]

    topology, positions, charge_molecules = build_openff_topology_from_instances(rdkit_mols)
    force_field = ForceField(FORCE_FIELD)
    interchange = force_field.create_interchange(topology, charge_from_molecules=charge_molecules)
    interchange.positions = positions
    interchange.box = box_vectors_nm * off_unit.nanometer
    openmm_system = interchange.to_openmm(combine_nonbonded_forces=True)
    openmm_topology = interchange.to_openmm_topology()
    openmm_topology.setPeriodicBoxVectors(box_vectors_nm * omm_unit.nanometer)
    integrator = LangevinMiddleIntegrator(OPENMM_MD_TEMPERATURE_K * omm_unit.kelvin, OPENMM_MD_FRICTION_PER_PS / omm_unit.picosecond, OPENMM_MD_TIMESTEP_FS * omm_unit.femtosecond)
    try:
        platform = openmm.Platform.getPlatformByName(OPENMM_PLATFORM_NAME)
        simulation = Simulation(openmm_topology, openmm_system, integrator, platform, OPENMM_PLATFORM_PROPERTIES)
    except Exception as exc:
        print(f"OpenMM {OPENMM_PLATFORM_NAME} unavailable; using default platform ({exc})")
        simulation = Simulation(openmm_topology, openmm_system, integrator)
    simulation.context.setPositions(interchange.positions.to_openmm())
    state0 = simulation.context.getState(getEnergy=True)
    print(f"Initial potential energy: {state0.getPotentialEnergy()}")
    simulation.minimizeEnergy(tolerance=OPENMM_MINIMIZATION_TOLERANCE_KJ_MOL_NM * omm_unit.kilojoule_per_mole / omm_unit.nanometer)
    state = simulation.context.getState(getEnergy=True, getPositions=True, enforcePeriodicBox=False)
    minimized_energy_kj_mol = state.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)
    print(f"Minimized potential energy: {state.getPotentialEnergy()}")
    if not np.isfinite(minimized_energy_kj_mol):
        raise RuntimeError(f"OpenMM minimization produced non-finite potential energy: {state.getPotentialEnergy()}")
    minimized_positions = state.getPositions(asNumpy=True)
    simulation.context.setPositions(minimized_positions)
    def write_packed_dcd_frame(dcd: DCDFile, state) -> float:
        positions_nm = state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
        box_vectors_nm_current = state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
        box_lengths_a = np.array([np.linalg.norm(vector) for vector in box_vectors_nm_current], dtype=float) * 10.0
        packed_nm = whole_packed_positions_a(positions_nm * 10.0, rdkit_mols, box_lengths_a, centered_input=False) * 0.1
        dcd.writeModel(packed_nm * omm_unit.nanometer, periodicBoxVectors=box_vectors_nm_current * omm_unit.nanometer)
        return state.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)

    def run_md_segment(label: str, total_steps: int, dcd_path: Path) -> tuple[object, float]:
        print(f"Running {label}: steps={total_steps}, target_frames={TRAJECTORY_FRAMES}")
        start = time.perf_counter()
        steps_done = 0
        last_state = None
        with dcd_path.open("wb") as handle:
            dcd = DCDFile(handle, openmm_topology, OPENMM_MD_TIMESTEP_FS * omm_unit.femtosecond, interval=max(1, total_steps // TRAJECTORY_FRAMES))
            for frame_idx in range(TRAJECTORY_FRAMES):
                target_step = int(round((frame_idx + 1) * total_steps / TRAJECTORY_FRAMES))
                steps = max(1, target_step - steps_done)
                simulation.step(steps)
                steps_done += steps
                state_i = simulation.context.getState(getEnergy=True, getPositions=True, getVelocities=True, enforcePeriodicBox=False)
                potential_kj_mol = state_i.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)
                positions_nm = state_i.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
                velocities_nm_ps = state_i.getVelocities(asNumpy=True).value_in_unit(omm_unit.nanometer / omm_unit.picosecond)
                if not (np.isfinite(potential_kj_mol) and np.all(np.isfinite(positions_nm)) and np.all(np.isfinite(velocities_nm_ps))):
                    raise RuntimeError(f"OpenMM {label} became unstable at step {steps_done}: potential={potential_kj_mol} kJ/mol")
                potential_kj_mol = write_packed_dcd_frame(dcd, state_i)
                last_state = state_i
                if (frame_idx + 1) % 10 == 0 or frame_idx == TRAJECTORY_FRAMES - 1:
                    print(f"{label} step {steps_done}/{total_steps}: potential={potential_kj_mol:.3f} kJ/mol")
        print(f"Completed {label} in {(time.perf_counter() - start) / 60.0:.2f} min; wrote {dcd_path}")
        return last_state, potential_kj_mol

    nvt_dcd_path = OUTPUT_ROOT / "pet_slab_openmm_nvt_whole_packed.dcd"
    npt_dcd_path = OUTPUT_ROOT / "pet_slab_openmm_npt_whole_packed.dcd"
    final_state, nvt_energy_kj_mol = run_md_segment("NVT", OPENMM_MD_STEPS, nvt_dcd_path)
    openmm_system.addForce(MonteCarloBarostat(OPENMM_NPT_PRESSURE_ATM * omm_unit.atmosphere, OPENMM_MD_TEMPERATURE_K * omm_unit.kelvin, OPENMM_BAROSTAT_FREQUENCY_STEPS))
    simulation.context.reinitialize(preserveState=True)
    final_state, npt_energy_kj_mol = run_md_segment("NPT", OPENMM_NPT_STEPS, npt_dcd_path)
    final_state = simulation.context.getState(getEnergy=True, getPositions=True, enforcePeriodicBox=False)
    final_energy_kj_mol = final_state.getPotentialEnergy().value_in_unit(omm_unit.kilojoule_per_mole)
    if not np.isfinite(final_energy_kj_mol):
        raise RuntimeError(f"OpenMM final MD frame has non-finite potential energy: {final_state.getPotentialEnergy()}")
    print(f"Final MD potential energy: {final_state.getPotentialEnergy()}")
    final_box_vectors_nm = final_state.getPeriodicBoxVectors(asNumpy=True).value_in_unit(omm_unit.nanometer)
    final_box_lengths_a = np.array([np.linalg.norm(vector) for vector in final_box_vectors_nm], dtype=float) * 10.0
    final_box_parameters = [float(final_box_lengths_a[0]), float(final_box_lengths_a[1]), float(final_box_lengths_a[2]), 0.0, 0.0, 0.0]
    final_positions_nm = final_state.getPositions(asNumpy=True).value_in_unit(omm_unit.nanometer)
    for mol in rdkit_mols:
        conf = mol.GetConformer()
        for atom_idx in range(mol.GetNumAtoms()):
            p = final_positions_nm[atom_idx]
            conf.SetAtomPosition(atom_idx, Point3D(float(10.0 * p[0]), float(10.0 * p[1]), float(10.0 * p[2])))
        final_positions_nm = final_positions_nm[mol.GetNumAtoms():]
    for mol in rdkit_mols:
        make_rdkit_molecule_whole(mol, final_box_parameters, centered_input=False)
    final_cif_path = OUTPUT_ROOT / "output.cif"
    write_rdkit_mols_to_pdbx(rdkit_mols, final_cif_path, box_vectors_nm=final_box_vectors_nm)
    manifest["openmm_md_final_cif"] = str(final_cif_path.relative_to(EXAMPLES_ROOT))
    manifest["openmm_minimized_potential_energy"] = str(state.getPotentialEnergy())
    manifest["openmm_nvt_dcd"] = str(nvt_dcd_path.relative_to(EXAMPLES_ROOT))
    manifest["openmm_npt_dcd"] = str(npt_dcd_path.relative_to(EXAMPLES_ROOT))
    manifest["openmm_nvt_duration_ns"] = OPENMM_MD_DURATION_NS
    manifest["openmm_npt_duration_ns"] = OPENMM_NPT_DURATION_NS
    manifest["openmm_md_timestep_fs"] = OPENMM_MD_TIMESTEP_FS
    manifest["openmm_nvt_final_potential_energy_kj_mol"] = nvt_energy_kj_mol
    manifest["openmm_npt_final_potential_energy"] = str(final_state.getPotentialEnergy())
    manifest["openmm_npt_final_box_nm"] = final_box_vectors_nm.tolist()
    manifest["openmm_md_stable"] = True
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    print(f"Wrote stable-MD final CIF to {final_cif_path}")
else:
    print("OpenMM minimization skipped. Set RUN_OPENMM_MINIMIZATION = True to generate the final minimized CIF.")